In [ ]:
# In[1]:

import pandas as pd

# Load CSV files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
metric['timestamp_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp_dt'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp_dt'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# Helper to build compact summary for a telemetry dataframe
def build_summary_df(df, name_col_for_counts=None):
    total_rows = len(df)
    earliest = df['timestamp_dt'].min() if total_rows>0 else pd.NaT
    latest = df['timestamp_dt'].max() if total_rows>0 else pd.NaT
    cmdb_list = df['cmdb_id'].dropna().unique().tolist()[:20]
    # counts for the name column (kpi/trace/log/message)
    if name_col_for_counts and name_col_for_counts in df.columns:
        name_counts = df[name_col_for_counts].value_counts().head(20)
        # represent as list of (name, count) tuples for compactness
        name_counts_list = list(name_counts.items())
    else:
        name_counts_list = []
    summary = pd.DataFrame([{
        'total_rows': total_rows,
        'earliest_timestamp_utc': earliest,
        'latest_timestamp_utc': latest,
        'unique_cmdb_ids_top20': cmdb_list,
        'top20_name_counts': name_counts_list
    }])
    return summary

# Build summaries
metric_summary_df = build_summary_df(metric, name_col_for_counts='kpi_name')
trace_summary_df = build_summary_df(trace, name_col_for_counts='trace_name')
log_summary_df = build_summary_df(log, name_col_for_counts='log_name')
error_logs_summary_df = build_summary_df(error_logs, name_col_for_counts='message')

# For metric.csv compute global P95 per (cmdb_id, kpi_name) using the full series
metric_p95 = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(p95_value=lambda s: s.quantile(0.95), count_of_points='count')
    .reset_index()
)
# Sort by p95 descending and keep top 20
metric_p95_top20 = metric_p95.sort_values('p95_value', ascending=False).head(20).reset_index(drop=True)

# Round p95_value for compact display
metric_p95_top20['p95_value'] = metric_p95_top20['p95_value'].round(6)

# Display compact summaries and top-20 p95 table
metric_summary_df, trace_summary_df, log_summary_df, error_logs_summary_df, metric_p95_top20

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry files (UTC):\n"
    "- metric.csv: 1900 rows; earliest 2024-01-19 21:14:00+00:00; latest 2024-01-19 21:38:00+00:00.\n"
    "  Example cmdb_ids (top): adservice, cartservice, checkoutservice, currencyservice, frontend, paymentservice, productcatalogservice, recommendationservice, shippingservice, redis, ...\n"
    "  Top metric names (counts): cpu (275), workload (275), socket (275), ...\n"
    "- trace.csv: 3776 rows; earliest 2024-01-19 21:14:00+00:00; latest 2024-01-19 21:38:00+00:00.\n"
    "  Example cmdb_ids (top): checkoutservice, currencyservice, emailservice, ...\n"
    "  Top trace names (examples): trace.from_root.duration_mean (172), trace.self.duration_mean, ...\n"
    "- log.csv: 458 rows; earliest 2024-01-19 21:14:00+00:00; latest 2024-01-19 21:38:00+00:00.\n"
    "  Example cmdb_ids (top): adservice, cartservice, checkoutservice, currencyservice, ...\n"
    "  Top log names (counts): log.error_count (229), log.row_count (229).\n"
    "- error_logs.csv: 0 rows (no error log entries).\n\n"
    "Metric P95 highlights (top pairs by global P95, using full series):\n"
    "  1) adservice - mem: p95 = 1.120347e+08 (25 points)\n"
    "  2) currencyservice - mem: p95 = 5.359667e+07 (25 points)\n"
    "  3) recommendationservice - mem: p95 = 4.601001e+07 (25 points)\n"
    "  4) paymentservice - mem: p95 = 4.569300e+07 (25 points)\n"
    "  5) cartservice - mem: p95 = 4.481481e+07 (25 points)\n"
    "  6) emailservice - mem: p95 = 4.399858e+07 (25 points)\n"
    "  7) frontend - mem: p95 = 1.704695e+07 (25 points)\n"
    "  8) checkoutservice - mem: p95 = 1.211376e+07 (25 points)\n"
    "  9) productcatalogservice - mem: p95 = 1.084882e+07 (25 points)\n"
    " 10) shippingservice - mem: p95 = 9.289400e+06 (25 points)\n"
    " 11) redis - mem: p95 = 3.013564e+06 (25 points)\n"
    " 12) adservice - diskio: p95 = 7.356591e+04 (25 points)\n"
    "\nInterpretation: memory (mem) KPIs dominate the highest P95 values across many services (adservice, currencyservice, recommendationservice, etc.), indicating unusually high memory observations for those components in the captured window. All metric series reported here have 25 data points each (consistent sampling). No error_logs entries are present to complement the metrics.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows    earliest_timestamp_utc      latest_timestamp_utc                              unique_cmdb_ids_top20                                  top20_name_counts
0        1900 2024-01-19 21:14:00+00:00 2024-01-19 21:38:00+00:00  [adservice, cartservice, checkoutservice, curr...  [(cpu, 275), (workload, 275), (socket, 275), (...,    total_rows    earliest_timestamp_utc      latest_timestamp_utc                              unique_cmdb_ids_top20                                  top20_name_counts
0        3776 2024-01-19 21:14:00+00:00 2024-01-19 21:38:00+00:00  [checkoutservice, currencyservice, emailservic...  [(trace.from_root.duration_mean, 172), (trace....,    total_rows    earliest_timestamp_utc      latest_timestamp_utc                              unique_cmdb_ids_top20                               top20_name_counts
0         458 2024-01-19 21:14:00+00:00 2024-01-19 21:38:00+00:00  [adservice, cartservice, checkoutservice, curr...  [(log.error_count, 229), (log.row_count, 229)],    total_rows earliest_timestamp_utc latest_timestamp_utc unique_cmdb_ids_top20 top20_name_counts
0           0                    NaT                  NaT                    []                [],                   cmdb_id  kpi_name     p95_value  count_of_points
0               adservice       mem  1.120347e+08               25
1         currencyservice       mem  5.359667e+07               25
2   recommendationservice       mem  4.601001e+07               25
3          paymentservice       mem  4.569300e+07               25
4             cartservice       mem  4.481481e+07               25
5            emailservice       mem  4.399858e+07               25
6                frontend       mem  1.704695e+07               25
7         checkoutservice       mem  1.211376e+07               25
8   productcatalogservice       mem  1.084882e+07               25
9         shippingservice       mem  9.289400e+06               25
10                  redis       mem  3.013564e+06               25
11              adservice    diskio  7.356591e+04               25
12                  redis    diskio  1.338194e+04               25
13  productcatalogservice  workload  1.511970e+02               25
14        currencyservice  workload  8.147375e+01               25
15        currencyservice       cpu  4.954323e+01               25
16               frontend    socket  2.900000e+01               25
17            cartservice  workload  2.660250e+01               25
18  recommendationservice  workload  1.990845e+01               25
19        currencyservice    socket  1.820333e+01               25)```
```

In [ ]:
# In[2]:

import pandas as pd
# reuse existing 'metric' variable loaded earlier; ensure timestamp_dt exists
if 'timestamp_dt' not in metric.columns:
    metric['timestamp_dt'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric series (before any filtering)
metric_p95_full = (
    metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(p95_value=lambda s: s.quantile(0.95), count_of_points='count')
    .reset_index()
)

# 2) Define the incident window (UTC)
window_start = pd.to_datetime("2024-01-19 21:11:59", utc=True)
window_end = pd.to_datetime("2024-01-19 21:41:59", utc=True)

# 3) Filter metric to the incident window (inclusive)
metric_window = metric[(metric['timestamp_dt'] >= window_start) & (metric['timestamp_dt'] <= window_end)].copy()

# 4) Join windowed data with global P95 info
metric_window = metric_window.merge(metric_p95_full, on=['cmdb_id', 'kpi_name'], how='left')

# 5) Detect anomalies in the window where value >= global P95
metric_window['is_anomaly'] = metric_window['value'] >= metric_window['p95_value']

# 6) Aggregate anomaly info per (cmdb_id, kpi_name)
anomaly_groups = (
    metric_window[metric_window['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        anomaly_count_in_window=('is_anomaly', 'size'),
        earliest_anomaly_timestamp_utc=('timestamp_dt', 'min'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 7) Merge with global p95 and full-series counts
anomaly_summary = anomaly_groups.merge(metric_p95_full, on=['cmdb_id', 'kpi_name'], how='left')

# 8) Compute ratio and reorder/round columns for compactness
anomaly_summary['max_value_to_p95_ratio'] = anomaly_summary['max_value_in_window'] / anomaly_summary['p95_value']
anomaly_summary = anomaly_summary[
    ['cmdb_id', 'kpi_name', 'p95_value', 'count_of_points', 'anomaly_count_in_window',
     'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'max_value_to_p95_ratio']
]

# 9) Sort by anomaly_count descending and keep top 20
anomalies_summary_top20 = anomaly_summary.sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

# 10) Compact formatting: round numeric columns for readability
anomalies_summary_top20['p95_value'] = anomalies_summary_top20['p95_value'].round(6)
anomalies_summary_top20['max_value_in_window'] = anomalies_summary_top20['max_value_in_window'].round(6)
anomalies_summary_top20['max_value_to_p95_ratio'] = anomalies_summary_top20['max_value_to_p95_ratio'].replace([pd.NA, pd.NaT], None).round(3)

# 11) Total anomalous points across all component-KPIs in the window
total_anomalous_points = int(metric_window['is_anomaly'].sum())

# 12) Distinct cmdb_id candidates with at least one anomaly (limit top 20)
anomalous_cmdbs_top20 = pd.Series(metric_window.loc[metric_window['is_anomaly'], 'cmdb_id'].unique()).head(20)

# Display compact results
anomalies_summary_top20, total_anomalous_points, anomalous_cmdbs_top20

```
Out[2]:
```
```python
# Build a plain-English summary string from previous results in memory
# Uses: anomalies_summary_top20, total_anomalous_points, anomalous_cmdbs_top20

top_table = anomalies_summary_top20.copy()

lines = []
lines.append("Anomaly detection summary (window 2024-01-19 21:11:59 to 2024-01-19 21:41:59 UTC):")
lines.append(f"- Total anomalous points found in window: {int(total_anomalous_points)}")
# distinct cmdb ids (top)
cmdb_list = anomalous_cmdbs_top20.astype(str).tolist()
lines.append(f"- Distinct components with ≥1 anomaly (top {len(cmdb_list)}): " + ", ".join(cmdb_list))

lines.append("- Top anomalous component-KPIs (sorted by anomaly count):")
for i, row in top_table.head(20).iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    p95 = row['p95_value']
    total_pts = int(row['count_of_points'])
    anom_cnt = int(row['anomaly_count_in_window'])
    first_ts = row['earliest_anomaly_timestamp_utc']
    max_val = row['max_value_in_window']
    ratio = row['max_value_to_p95_ratio']
    # Format timestamp nicely if not null
    first_ts_str = first_ts.isoformat() if pd.notna(first_ts) else "N/A"
    ratio_str = f"{ratio:.3f}" if pd.notna(ratio) else "N/A"
    lines.append(f"  {i+1}) {cmdb} - {kpi}: anomalies={anom_cnt}, p95={p95}, points_in_full_series={total_pts}, first_anomaly={first_ts_str}, max_in_window={max_val}, max/p95={ratio_str}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  p95_value  count_of_points  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  max_value_to_p95_ratio
0               adservice       error   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
1               adservice      socket   4.000000               25                       25      2024-01-19 21:14:00+00:00             4.000000                   1.000
2             cartservice      diskio   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
3   recommendationservice      diskio   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
4          paymentservice      diskio   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
5            emailservice      socket   3.000000               25                       25      2024-01-19 21:14:00+00:00             3.000000                   1.000
6                frontend       error   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
7                frontend      diskio   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
8            emailservice      diskio   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
9         currencyservice      diskio   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
10        checkoutservice      socket   9.000000               25                       25      2024-01-19 21:14:00+00:00             9.000000                   1.000
11            cartservice      socket   4.000000               25                       25      2024-01-19 21:14:00+00:00             4.000000                   1.000
12  productcatalogservice      diskio   0.000000               25                       25      2024-01-19 21:14:00+00:00             0.000000                     NaN
13  recommendationservice      socket   4.000000               25                       25      2024-01-19 21:14:00+00:00             4.000000                   1.000
14        shippingservice      socket   2.000000               25                       25      2024-01-19 21:14:00+00:00             2.000000                   1.000
15                  redis      socket   4.000000               25                       25      2024-01-19 21:14:00+00:00             4.000000                   1.000
16         paymentservice      socket   3.000000               25                       25      2024-01-19 21:14:00+00:00             3.000000                   1.000
17  productcatalogservice      socket   3.000000               25                       25      2024-01-19 21:14:00+00:00             3.000000                   1.000
18               frontend      socket  29.000000               25                        9      2024-01-19 21:14:00+00:00            29.000000                   1.000
19            cartservice  latency-90   0.008828               25                        2      2024-01-19 21:28:00+00:00             0.045571                   5.162, 573, 0                 adservice
1               cartservice
2           checkoutservice
3           currencyservice
4              emailservice
5                  frontend
6            paymentservice
7     productcatalogservice
8     recommendationservice
9                     redis
10          shippingservice
11        frontend-external
dtype: object)```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse existing 'trace' variable loaded earlier; ensure timestamp_dt exists
if 'timestamp_dt' not in trace.columns:
    trace['timestamp_dt'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full trace series
trace_p95_full = (
    trace.groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(p95_value=lambda s: s.quantile(0.95), count_of_points='count')
    .reset_index()
)

# 2) Incident window (UTC)
window_start = pd.to_datetime("2024-01-19 21:11:59", utc=True)
window_end = pd.to_datetime("2024-01-19 21:41:59", utc=True)

# 3) Filter trace to the incident window (inclusive)
trace_window = trace[(trace['timestamp_dt'] >= window_start) & (trace['timestamp_dt'] <= window_end)].copy()

# 4) Merge windowed data with global P95 info
trace_window = trace_window.merge(trace_p95_full, on=['cmdb_id', 'trace_name'], how='left')

# 5) Detect anomalies in the window where value >= global P95
trace_window['is_anomaly'] = trace_window['value'] >= trace_window['p95_value']

# 6) Aggregate anomaly info per (cmdb_id, trace_name)
trace_anomaly_groups = (
    trace_window[trace_window['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        anomaly_count_in_window=('is_anomaly', 'size'),
        earliest_anomaly_timestamp_utc=('timestamp_dt', 'min'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 7) Merge with global p95 and full-series counts
trace_anomaly_summary = trace_anomaly_groups.merge(trace_p95_full, on=['cmdb_id', 'trace_name'], how='left')

# 8) Compute ratio and select columns
trace_anomaly_summary['max_value_to_p95_ratio'] = trace_anomaly_summary['max_value_in_window'] / trace_anomaly_summary['p95_value']

trace_anomaly_summary = trace_anomaly_summary[
    ['cmdb_id', 'trace_name', 'p95_value', 'count_of_points', 'anomaly_count_in_window',
     'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'max_value_to_p95_ratio']
]

# 9) Sort by ratio descending and keep top 20
anomalies_summary_top20_trace = trace_anomaly_summary.sort_values('max_value_to_p95_ratio', ascending=False).head(20).reset_index(drop=True)

# 10) Compact formatting: round numeric columns for readability
anomalies_summary_top20_trace['p95_value'] = anomalies_summary_top20_trace['p95_value'].round(6)
anomalies_summary_top20_trace['max_value_in_window'] = anomalies_summary_top20_trace['max_value_in_window'].round(6)
anomalies_summary_top20_trace['max_value_to_p95_ratio'] = anomalies_summary_top20_trace['max_value_to_p95_ratio'].replace([pd.NA, pd.NaT, float('inf')], pd.NA).round(3)

# 11) Total anomalous points across all trace component-KPIs in the window
total_trace_anomalous_points = int(trace_window['is_anomaly'].sum())

# 12) Distinct cmdb_id candidates with at least one trace anomaly (limit top 20)
anomalous_trace_cmdbs_top20 = pd.Series(trace_window.loc[trace_window['is_anomaly'], 'cmdb_id'].unique()).head(20)

# Display compact results
anomalies_summary_top20_trace, total_trace_anomalous_points, anomalous_trace_cmdbs_top20

```
Out[3]:
```
```python
# Build a plain-English summary string from the trace anomaly results in memory
# Uses: anomalies_summary_top20_trace, total_trace_anomalous_points, anomalous_trace_cmdbs_top20

top = anomalies_summary_top20_trace.copy()
lines = []
lines.append("Trace anomaly summary (global P95 vs window 2024-01-19 21:11:59 to 2024-01-19 21:41:59 UTC):")
lines.append(f"- Total trace-anomalous points in window: {int(total_trace_anomalous_points)}")
cmdb_list = anomalous_trace_cmdbs_top20.astype(str).tolist()
lines.append(f"- Components with ≥1 trace anomaly (top {len(cmdb_list)}): " + ", ".join(cmdb_list))
lines.append("- Top trace anomalies by max_value_to_p95_ratio:")

# show up to top 10 for compactness (results already limited to 20)
n_show = min(10, len(top))
for i in range(n_show):
    row = top.iloc[i]
    cmdb = row['cmdb_id']
    tname = row['trace_name']
    p95 = row['p95_value']
    total_pts = int(row['count_of_points'])
    anom_cnt = int(row['anomaly_count_in_window'])
    first_ts = row['earliest_anomaly_timestamp_utc']
    first_ts_str = first_ts.isoformat() if pd.notna(first_ts) else "N/A"
    max_val = row['max_value_in_window']
    ratio = row['max_value_to_p95_ratio']
    ratio_str = f"{ratio:.3f}" if pd.notna(ratio) else "N/A"
    lines.append(f"  {i+1}) {cmdb} | {tname} — anomalies={anom_cnt}, p95={p95}, points_full={total_pts}, first={first_ts_str}, max_in_window={max_val}, max/p95={ratio_str}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                      trace_name  p95_value  count_of_points  anomaly_count_in_window earliest_anomaly_timestamp_utc  max_value_in_window  max_value_to_p95_ratio
0   recommendationservice                   trace.from_root.duration_mean   0.000066               25                        2      2024-01-19 21:14:00+00:00             0.001332                  20.251
1                    root    trace.to_recommendationservice.duration_mean   0.000066               25                        2      2024-01-19 21:14:00+00:00             0.001332                  20.251
2                    root     trace.to_recommendationservice.duration_p95   0.000178               25                        2      2024-01-19 21:14:00+00:00             0.003528                  19.850
3   recommendationservice                    trace.from_root.duration_p95   0.000178               25                        2      2024-01-19 21:14:00+00:00             0.003528                  19.850
4         checkoutservice           trace.to_currencyservice.duration_p95   0.000877               25                        2      2024-01-19 21:33:00+00:00             0.001291                   1.472
5         currencyservice         trace.from_checkoutservice.duration_p95   0.000877               25                        2      2024-01-19 21:33:00+00:00             0.001291                   1.472
6         frontendservice           trace.to_checkoutservice.duration_p95   1.084670               25                        2      2024-01-19 21:27:00+00:00             1.532339                   1.413
7         checkoutservice         trace.from_frontendservice.duration_p95   1.084670               25                        2      2024-01-19 21:27:00+00:00             1.532339                   1.413
8   recommendationservice    trace.to_recommendationservice.duration_mean   0.004383               25                        2      2024-01-19 21:36:00+00:00             0.006188                   1.412
9   recommendationservice  trace.from_recommendationservice.duration_mean   0.004383               25                        2      2024-01-19 21:36:00+00:00             0.006188                   1.412
10        checkoutservice          trace.to_currencyservice.duration_mean   0.002196               25                        2      2024-01-19 21:27:00+00:00             0.003062                   1.394
11        currencyservice        trace.from_checkoutservice.duration_mean   0.002196               25                        2      2024-01-19 21:27:00+00:00             0.003062                   1.394
12        checkoutservice    trace.to_productcatalogservice.duration_mean   0.000019               25                        2      2024-01-19 21:16:00+00:00             0.000025                   1.341
13  productcatalogservice        trace.from_checkoutservice.duration_mean   0.000019               25                        2      2024-01-19 21:16:00+00:00             0.000025                   1.341
14           emailservice        trace.from_checkoutservice.duration_mean   0.000339               25                        2      2024-01-19 21:14:00+00:00             0.000455                   1.341
15        checkoutservice             trace.to_emailservice.duration_mean   0.000339               25                        2      2024-01-19 21:14:00+00:00             0.000455                   1.341
16         paymentservice                    trace.from_root.duration_p95   0.000116               24                        2      2024-01-19 21:18:00+00:00             0.000155                   1.337
17                   root            trace.to_paymentservice.duration_p95   0.000116               24                        2      2024-01-19 21:18:00+00:00             0.000155                   1.337
18        frontendservice          trace.to_checkoutservice.duration_mean   0.457372               25                        2      2024-01-19 21:35:00+00:00             0.601650                   1.315
19        checkoutservice        trace.from_frontendservice.duration_mean   0.457372               25                        2      2024-01-19 21:35:00+00:00             0.601650                   1.315, 1390, 0          checkoutservice
1          currencyservice
2             emailservice
3          frontendservice
4           paymentservice
5    productcatalogservice
6    recommendationservice
7                     root
dtype: object)```
```

In [ ]:
# In[4]:

import pandas as pd

# Ensure log timestamps parsed
if 'timestamp_dt' not in log.columns:
    log['timestamp_dt'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)

# Candidate cmdb_ids and incident window (UTC)
candidates = ['cartservice', 'checkoutservice', 'currencyservice', 'frontendservice', 'recommendationservice']
window_start = pd.to_datetime("2024-01-19 21:11:59", utc=True)
window_end = pd.to_datetime("2024-01-19 21:41:59", utc=True)

# Filter log rows to window and candidates
log_window = log[
    (log['timestamp_dt'] >= window_start) &
    (log['timestamp_dt'] <= window_end) &
    (log['cmdb_id'].isin(candidates))
].copy()

# Part 1: per-cmdb_id compact summary
# Basic counts and timestamps
grouped = log_window.groupby('cmdb_id', sort=False).agg(
    row_count_in_window=('cmdb_id', 'size'),
    earliest_timestamp_in_window=('timestamp_dt', 'min'),
    latest_timestamp_in_window=('timestamp_dt', 'max')
).reset_index()

# Error count sums and max (for log_name == 'log.error_count')
error_stats = (
    log_window[log_window['log_name'] == 'log.error_count']
    .groupby('cmdb_id')['value']
    .agg(sum_log_error_count_in_window='sum', max_log_error_count_in_window='max')
    .reset_index()
)

# Distinct log names (limit to 20 per cmdb)
distinct_names = (
    log_window.groupby('cmdb_id')['log_name']
    .agg(lambda s: list(pd.Series(s.unique())[:20]))  # keep up to 20 unique names
    .reset_index()
    .rename(columns={'log_name': 'distinct_log_names_list'})
)

# Merge summaries
summary_df = grouped.merge(error_stats, on='cmdb_id', how='left').merge(distinct_names, on='cmdb_id', how='left')

# Fill NaN sums/max with 0 for cmdbs that had no log.error_count rows
summary_df['sum_log_error_count_in_window'] = summary_df['sum_log_error_count_in_window'].fillna(0)
summary_df['max_log_error_count_in_window'] = summary_df['max_log_error_count_in_window'].fillna(0)

# Keep columns in requested order and compact display
summary_df = summary_df[
    ['cmdb_id', 'row_count_in_window', 'sum_log_error_count_in_window', 'max_log_error_count_in_window',
     'earliest_timestamp_in_window', 'latest_timestamp_in_window', 'distinct_log_names_list']
].reset_index(drop=True)

# Part 2: up to 20 raw log rows where log_name contains 'error' OR value > 0, sorted by timestamp ascending
mask_error_name = log_window['log_name'].astype(str).str.contains('error', case=False, na=False)
mask_value_positive = pd.to_numeric(log_window['value'], errors='coerce') > 0
raw_log_rows = log_window[mask_error_name | mask_value_positive].copy()

# Select requested columns and sort
raw_log_rows = raw_log_rows[['timestamp_dt', 'cmdb_id', 'log_name', 'value']].sort_values('timestamp_dt', ascending=True).head(20).reset_index(drop=True)

# Rename timestamp column to be explicit as UTC datetime
raw_log_rows = raw_log_rows.rename(columns={'timestamp_dt': 'timestamp_utc'})

# Display compact results
summary_df, raw_log_rows

```
Out[4]:
```
```python
summary = (
    "Log analysis for window 2024-01-19 21:11:59–21:41:59 UTC (candidates: cartservice, checkoutservice, "
    "currencyservice, frontendservice, recommendationservice):\n"
    "- Per-component summary (rows in window):\n"
    "  * cartservice: 50 rows; sum(log.error_count)=0; max(log.error_count)=0; log names=[log.error_count, log.row_count]\n"
    "  * checkoutservice: 50 rows; sum(log.error_count)=0; max(log.error_count)=0; log names=[log.error_count, log.row_count]\n"
    "  * currencyservice: 50 rows; sum(log.error_count)=0; max(log.error_count)=0; log names=[log.error_count, log.row_count]\n"
    "  * recommendationservice: 50 rows; sum(log.error_count)=0; max(log.error_count)=0; log names=[log.error_count, log.row_count]\n"
    "  * frontendservice: no rows in the window for the given filters\n"
    "- Interpretation: None of the candidate services recorded any log.error_count > 0 in the window (all sum/max = 0).\n"
    "- However, there are notable spikes in log.row_count (indicating traffic/row volumes) at these timestamps (UTC):\n"
    "  * 2024-01-19T21:15:00 — currencyservice log.row_count = 2567\n"
    "  * 2024-01-19T21:15:00 — cartservice log.row_count = 786\n"
    "  * 2024-01-19T21:15:00 — recommendationservice log.row_count = 563\n"
    "  * multiple checkoutservice row_count entries (e.g., 78 at 21:15, 99 at 21:16)\n"
    "- Conclusion: No explicit error logs were produced by these candidate services during the incident window, "
    "but several services (currencyservice, cartservice, recommendationservice, checkoutservice) show large row_count spikes "
    "around 21:15 UTC which may warrant investigation (traffic/throughput or downstream effects)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id  row_count_in_window  sum_log_error_count_in_window  max_log_error_count_in_window earliest_timestamp_in_window latest_timestamp_in_window           distinct_log_names_list
0            cartservice                   50                              0                              0    2024-01-19 21:14:00+00:00  2024-01-19 21:38:00+00:00  [log.error_count, log.row_count]
1        checkoutservice                   50                              0                              0    2024-01-19 21:14:00+00:00  2024-01-19 21:38:00+00:00  [log.error_count, log.row_count]
2        currencyservice                   50                              0                              0    2024-01-19 21:14:00+00:00  2024-01-19 21:38:00+00:00  [log.error_count, log.row_count]
3  recommendationservice                   50                              0                              0    2024-01-19 21:14:00+00:00  2024-01-19 21:38:00+00:00  [log.error_count, log.row_count],                timestamp_utc                cmdb_id         log_name  value
0  2024-01-19 21:14:00+00:00            cartservice  log.error_count      0
1  2024-01-19 21:14:00+00:00            cartservice    log.row_count     11
2  2024-01-19 21:14:00+00:00        checkoutservice  log.error_count      0
3  2024-01-19 21:14:00+00:00        checkoutservice    log.row_count      3
4  2024-01-19 21:14:00+00:00        currencyservice  log.error_count      0
5  2024-01-19 21:14:00+00:00        currencyservice    log.row_count     40
6  2024-01-19 21:14:00+00:00  recommendationservice  log.error_count      0
7  2024-01-19 21:14:00+00:00  recommendationservice    log.row_count      9
8  2024-01-19 21:15:00+00:00  recommendationservice    log.row_count    563
9  2024-01-19 21:15:00+00:00  recommendationservice  log.error_count      0
10 2024-01-19 21:15:00+00:00        currencyservice  log.error_count      0
11 2024-01-19 21:15:00+00:00        currencyservice    log.row_count   2567
12 2024-01-19 21:15:00+00:00        checkoutservice  log.error_count      0
13 2024-01-19 21:15:00+00:00            cartservice    log.row_count    786
14 2024-01-19 21:15:00+00:00            cartservice  log.error_count      0
15 2024-01-19 21:15:00+00:00        checkoutservice    log.row_count     78
16 2024-01-19 21:16:00+00:00            cartservice  log.error_count      0
17 2024-01-19 21:16:00+00:00            cartservice    log.row_count    785
18 2024-01-19 21:16:00+00:00        checkoutservice  log.error_count      0
19 2024-01-19 21:16:00+00:00        checkoutservice    log.row_count     99)```
```